In [1]:
import pandas as pd
import numpy as np
import nltk
import re
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

lemmatizer=WordNetLemmatizer()

In [2]:
from nltk.tokenize import word_tokenize
from nltk.tokenize import sent_tokenize

In [3]:
dataset=pd.read_csv('/home/hammadali08/Personal/CSV file/IMDB Dataset.csv')
dataset

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [4]:
dataset.shape

(50000, 2)

In [5]:
dataset['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [8]:
stopwords = set([w.lower() for w in stopwords.words('english')])
corpus=[]
for i in range(0,len(dataset)):
    review = re.sub('[^a-zA-Z0-9]', ' ', dataset['review'][i])
    review = review.lower()
    review = review.split()

    review = [lemmatizer.lemmatize(word) for word in review if word not in stopwords]
    review = ' '.join(review)
    corpus.append(review)

# BOW

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
cv=CountVectorizer(max_features=2500,ngram_range=(1,2))
X=cv.fit_transform(corpus).toarray()

In [8]:
Y=pd.get_dummies(dataset['sentiment'],drop_first=True)
Y=Y.astype('int64')
Y.head()

,positive
0,1
1,1
2,1
3,0
4,1


In [9]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.1)

In [10]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100)
rf.fit(x_train,y_train)

/home/hammadali08/.local/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestClassifier()

In [11]:
rf.score(x_test,y_test),rf.score(x_train,y_train)

(0.8494, 1.0)

In [12]:
from sklearn.metrics import classification_report,confusion_matrix, accuracy_score
print(classification_report(y_test,rf.predict(x_test)))

              precision    recall  f1-score   support

           0       0.85      0.85      0.85      2483
           1       0.85      0.85      0.85      2517

    accuracy                           0.85      5000
   macro avg       0.85      0.85      0.85      5000
weighted avg       0.85      0.85      0.85      5000



In [13]:
confusion_matrix(y_test,rf.predict(x_test))

array([[2114,  369],
       [ 384, 2133]])

In [14]:
accuracy_score(y_test,rf.predict(x_test))

0.8494

# TF-IDF

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf=TfidfVectorizer(max_features=2500,ngram_range=(1,2))
X1=tf.fit_transform(corpus).toarray()

In [ ]:
x1_train,x1_test,y_train,y_test = train_test_split(X1,Y,test_size=0.1)

In [17]:
rf.fit(x1_train,y_train)

/home/hammadali08/.local/lib/python3.12/site-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestClassifier()

In [18]:
rf.score(x1_test,y_test)

0.8418

In [19]:
print(classification_report(y_test,rf.predict(x1_test)))

              precision    recall  f1-score   support

           0       0.84      0.85      0.84      2539
           1       0.84      0.84      0.84      2461

    accuracy                           0.84      5000
   macro avg       0.84      0.84      0.84      5000
weighted avg       0.84      0.84      0.84      5000



In [20]:
confusion_matrix(y_test,rf.predict(x1_test))

array([[2148,  391],
       [ 400, 2061]])

# With Word2Vec

In [9]:
from gensim.utils import simple_preprocess
words=[]
for sent in corpus:
    sent_token=sent_tokenize(sent)
    for sent in sent_token:
        words.append(simple_preprocess(sent))

In [10]:
import gensim
model=gensim.models.word2vec.Word2Vec(words,window=5,min_count=1)

In [11]:
model.epochs

5

In [12]:
def word2vec_avg(doc):
    return np.mean([model.wv[word]for word in doc if word in model.wv.index_to_key],axis=0)

In [13]:
from tqdm import tqdm
#apply for the entire sentences
import numpy as np
X=[]
for i in tqdm(range(len(words))):
    X.append(word2vec_avg(words[i]))

100%|██████████| 50000/50000 [12:07<00:00, 68.76it/s] 


In [20]:
import numpy as np
X=np.array(X)
X.ndim

2

In [ ]:
df_list = []
for i in range(len(X)):
    df_list.append(pd.DataFrame(X[i].reshape(1, -1)))

df = pd.concat(df_list, ignore_index=True)

/tmp/ipykernel_26227/1423230961.py:5: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(df_list, ignore_index=True)
